In [ ]:
!pip -q install langchain langchain_community pypdf fastembed chromadb

In [ ]:
# get the huggingface token
import os
from getpass import getpass
from google.colab import userdata

HF_TOKEN = getpass("Huggingface Token : ")
os.environ['HUGGINGFACEHUB_API_TOKEN'] = userdata.get('hugging_face_space')

Huggingface Token : ··········


In [ ]:
# import PDF reader
from langchain_community.document_loaders.pdf import PyPDFLoader

# load the document
# Source credits: https://ncert.nic.in/textbook/pdf/lekl101.pdf
loader = PyPDFLoader("/content/short_stories.pdf")
data = loader.load()

In [ ]:
# import text splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
chunk_size = 512,
chunk_overlap = 0,
)

chunks = text_splitter.split_documents(data)


In [ ]:
len(chunks)

46

In [ ]:
chunks[0]

Document(metadata={'source': '/content/short_stories.pdf', 'page': 0, 'page_label': '1'}, page_content='1/I SELL MY DREAMS\nShort stories\nINTRODUCTION\nA short story is a prose narrative of limited length.\nIt organises the action and thoughts of its\ncharacters into the pattern of a plot. The plot\nform may be comic, tragic, romantic or satiric.\nThe central incident is selected to manifest, as\nmuch as possible, the protagonist’s life and\ncharacter, and the details contribute to the\ndevelopment of the plot.\nThe term ‘short story’ covers a great diversity of\nprose fiction, right from the really short ‘short')

In [ ]:
from langchain_community.embeddings.fastembed import FastEmbedEmbeddings

model_name = "thenlper/gte-large"
embedding_model = FastEmbedEmbeddings(model_name="thenlper/gte-large")


In [ ]:
from langchain_community.vectorstores import Chroma

# initialize the vector store (save to disk)
db = Chroma.from_documents(chunks, embedding_model, persist_directory="./chroma_db")

In [ ]:
# Let's define the query, since we are going to use it multiple times.
query = "what is importance of I sell my dreams according to the author"

In [ ]:
# retrieve from vector db (load from disk) with query
retrieved_docs = db.similarity_search(query)
print(retrieved_docs[0].page_content)


10/KALEIDOSCOPE
Language Work
A. Vocabulary
Look up the meanings of the following phrases under ‘dream’
and ‘sell’ in the dictionary
dream         sell
dream on sell-by date
dream something away selling-point
(not) dream of doing something sell-out
dream something up selling price
look like a dream seller’s market
B. Grammar: Emphasis
Read this sentence carefully
One morning  at nine o’clock, while we were having
breakfast on the terrace of the Havana Riviera Hotel


In [ ]:
# initialize the retriever
retriever = db.as_retriever(
search_type="mmr", #similarity
search_kwargs={'k': 4}
)

In [ ]:
# import the question-answering chain and Huggingface Hub LLM
from langchain_community.llms import HuggingFaceHub

# define the llm
llm = HuggingFaceHub(repo_id="mistralai/Mistral-7B-Instruct-v0.2",
model_kwargs={
"temperature":0.1,
"max_new_tokens":512,
"return_full_text":False,
"repetition_penalty":1.1,
"top_p":0.9
})


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

template = """
<s>[INST]
You are an AI Assistant that follows instructions extremely well.
Please be truthful and give direct answers. Please tell 'I don't know' if user query is not in CONTEXT
[/INST]
CONTEXT: {context}
</s>
[INST]
{query}
[/INST]
"""


In [ ]:
prompt = ChatPromptTemplate.from_template(template)
output_parser = StrOutputParser()

In [ ]:
chain = (
{"context": retriever, "query": RunnablePassthrough()}
| prompt
| llm
| output_parser
)


In [ ]:
response = chain.invoke(query)
response

'The text "I Sell My Dreams" by Graham Greene does not directly discuss the importance of the phrase "I sell my dreams" according to the author. The text primarily focuses on the character of Frau Frieda and her mysterious ability to predict future events through her dreams. The phrase "I sell my dreams" is mentioned in relation to Frau Frieda when she is hired by a lady based on her statement that she "dreams." However, the text does not provide any insight into why the author might find it important for Frau Frieda to sell her dreams or what significance it holds within the context of the story.'

In [ ]:
response3 = chain.invoke("for how many years does the Author refer the lady to leave right away and not come back to Vienna")
response3

'The text mentions that thirteen years have passed since the first meeting between the narrator and Frau Frieda when she tells him he can go back to Vienna (Document 1, page 6). Therefore, the author refers to the lady leaving Vienna for thirteen years.'